# Practice 8. Negative selection on the Tennessee Eastman process

Industrial AI, KBTU. Goes with Lectures 12 and 13 and `inclass_work_13.ipynb`.
**1.25 points.**

**The process.** The Tennessee Eastman process (TEP) is a simulated chemical plant: a reactor, a
condenser, a separator, a recycle compressor and a stripper, under closed-loop control. It has
been the standard benchmark for fault detection since 1993, and you met it in Practice 2. One
sample every 3 minutes, 52 variables: 41 measurements `XMEAS1`–`XMEAS41` and 11 valve positions
`XMV1`–`XMV11`.

**The data** are R. Braatz's simulation files on Kaggle (`red2yang/tennessee-eastman-profbraatz`);
the loader cell downloads them once.

| file | samples | what it is |
|---|---|---|
| `d00.dat` | 500 | normal operation, 25 h. **The only training data** |
| `d00_te.dat` | 960 | another normal run, 48 h. Used to set $r_s$ and the alarm thresholds |
| `dXX_te.dat` | 960 | a test run with fault XX: 8 h normal, then the fault starts at sample 161 (index 160) and lasts 40 h |

The ten faults of this practice:

| fault | what happens | type |
|---|---|---|
| 1 | A/C feed ratio, B composition constant (stream 4) | step |
| 2 | B composition, A/C ratio constant (stream 4) | step |
| 4 | reactor cooling water inlet temperature | step |
| 5 | condenser cooling water inlet temperature | step |
| 6 | A feed loss (stream 1) | step |
| 7 | C header pressure loss (stream 4) | step |
| 11 | reactor cooling water inlet temperature | random variation |
| 3 | D feed temperature (stream 2) | step |
| 9 | D feed temperature (stream 2) | random variation |
| 15 | condenser cooling water valve | sticking |

**The question.** An immune system learns what "self" is and raises an alarm on anything else. It
never sees a fault while it learns. Can such a detector, trained on one normal day of the plant,
catch these faults? How fast, how often by mistake, and is it better than Isolation Forest or a
One-Class SVM?

### How the tasks are built

Each task says what to write: the functions, their arguments and what they return, and sometimes
a variable with a result. How you write them is up to you, as long as you use the method the task
names. Under your cell sits a cell of **tests**. Run it after your code: it either prints
`all tests passed` or stops at the first check that failed and says what is wrong. The tests check
what your code does on small cases with known answers, not how it is written.

These tests are public. When you hand in, the same tests run again together with hidden ones, on
other inputs, and the share of tests passed gives the points for the code. Code that only fits the
public cases loses those points.

Cells marked *Common ground* are written for you. Run them as they are. They fix the variables,
the random candidates and the features, so that everybody's numbers can be compared. Some result
variables need a common cell that comes after your functions; the task says so, and you write
them in the cell for the questions.

The methods are yours: the unit cube, negative selection with constant and variable radius, the
ROC curve and its area, PCA with the residual, the alarm threshold and the detection delay.
Isolation Forest and One-Class SVM come from `sklearn`: they are the baselines.
`sklearn.metrics.roc_auc_score` is for checking your AUC only.

### What you hand in

Two files:

1. this notebook, run top to bottom, named `Surname_Name_practice_8.ipynb`;
2. `practice_8_answers.md` with your answers to the questions under each task, named
   `Surname_Name_practice_8_answers.md`.

Every answer needs the numbers your notebook printed. A number the notebook does not print scores
zero, and so does a number printed with nothing said about it.

| Task | 1 | 2 | 3 | 4 | 5 | 6 | 7 | total |
|---|---|---|---|---|---|---|---|---|
| code (tests) | 0.05 | 0.10 | 0.08 | 0.10 | 0.07 | 0.10 | 0.10 | 0.60 |
| answers | 0.05 | 0.10 | 0.07 | 0.10 | 0.08 | 0.10 | 0.15 | 0.65 |

### The rules

- **Detectors learn from normal data only**: `X_train`, the 500 samples of `d00.dat`. No fault run
  may be used to fit anything: scaler, PCA, radius, detectors or threshold.
- **Thresholds and radii are set on `X_normal`**, the held-out normal run `d00_te.dat`.
- The first 160 samples of every test run are normal. They are the negatives of the ROC and the
  place where false alarms are counted.
- Do not rename the functions and variables the tests use, and do not edit the test cells.
- Before handing in, restart the kernel and run all cells; the whole notebook runs in about two
  minutes on a laptop. Then check your answers against the new output.
- Figures need axis labels and a title that says what to look at.
- Stuck? Part 2 of `inclass_work_13.ipynb` goes through negative selection step by step in 2-D.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import sklearn

BLUE, ORANGE, RED, GREEN, PURPLE, GREY = (
    "#2a6f8e", "#e07b39", "#c0392b", "#4a8f5a", "#9b3f9b", "#9aa0a6")

mpl.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e6e6e6",
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.labelsize": 10,
    "font.size": 10,
    "font.family": ["Segoe UI", "DejaVu Sans", "sans-serif"],
    "legend.frameon": False,
    "lines.linewidth": 1.6,
})

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

KAGGLE = {
    'tep': 'red2yang/tennessee-eastman-profbraatz',
}


def dataset_folder(name):
    """The folder with the files of a data set: the Kaggle copy (downloaded once, then cached),
    or data/<name>/ of the course repo if Kaggle is out of reach."""
    if KAGGLE[name]:
        try:
            import contextlib, io, logging, kagglehub
            logging.getLogger("kagglehub").setLevel(logging.ERROR)
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                folder = Path(kagglehub.dataset_download(KAGGLE[name]))
            print(f"{name}: Kaggle {KAGGLE[name]}")
            return folder
        except Exception as exc:
            print(f"Kaggle unavailable ({type(exc).__name__}), looking for data/{name}/")
    for root in [Path.cwd(), *Path.cwd().parents]:
        if (root / "data" / name).is_dir():
            return root / "data" / name
    raise FileNotFoundError(f"no data/{name}/ here or above; run data/fetch.py in the course repo")

print(" | ".join(f"{m} {sys.modules[m].__version__}" for m in
                 ("numpy", "pandas", "matplotlib", "scipy", "sklearn", "torch") if m in sys.modules))

numpy 2.5.5 | pandas 3.1.1 | matplotlib 3.11.2 | scipy 1.18.1 | sklearn 1.9.1


In [2]:
XMEAS = [f"XMEAS{i}" for i in range(1, 42)]  # 41 measured variables
XMV = [f"XMV{i}" for i in range(1, 12)]      # 11 manipulated variables: valve positions
VARS = XMEAS + XMV
HARD = [3, 9, 15]                            # known as nearly undetectable in these data
FAULTS = [1, 2, 4, 5, 6, 7, 11] + HARD
ONSET = 160                                  # first faulty sample of a test run, 0-based
MIN_PER_SAMPLE = 3

TEP = next(dataset_folder("tep").rglob("d00_te.dat")).parent


def read_tep(name):
    a = np.loadtxt(TEP / name)
    print(f"  {name:11s} as stored: {a.shape}")
    # one of the files is stored on its side
    if a.shape[0] == len(VARS):
        a = a.T
    return a


X_train = read_tep("d00.dat")        # normal operation: the self set
X_normal = read_tep("d00_te.dat")    # another normal run: thresholds and r_s
RUNS = {f: read_tep(f"d{f:02d}_te.dat") for f in FAULTS}
# 0 before the fault, 1 after; the fault runs are as long as d00_te
LABEL = (np.arange(len(X_normal)) >= ONSET).astype(int)


def dist_matrix(A, B):
    # Euclidean distance between every row of A and every row of B,
    # shape (len(A), len(B)).
    sq = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T
    return np.sqrt(np.maximum(sq, 0))


def draw_candidates(dim, n=5000, seed=13):
    # The random candidate detectors, the same for everybody: uniform in the unit cube.
    return np.random.default_rng(seed).random((n, dim))


assert all(len(r) == len(X_normal) for r in RUNS.values())
print("training", X_train.shape, "| held-out normal", X_normal.shape,
      "| fault runs", len(RUNS))

tep: Kaggle red2yang/tennessee-eastman-profbraatz
  d00.dat     as stored: (52, 500)
  d00_te.dat  as stored: (960, 52)
  d01_te.dat  as stored: (960, 52)
  d02_te.dat  as stored: (960, 52)
  d04_te.dat  as stored: (960, 52)
  d05_te.dat  as stored: (960, 52)
  d06_te.dat  as stored: (960, 52)
  d07_te.dat  as stored: (960, 52)
  d11_te.dat  as stored: (960, 52)
  d03_te.dat  as stored: (960, 52)
  d09_te.dat  as stored: (960, 52)
  d15_te.dat  as stored: (960, 52)
training (500, 52) | held-out normal (960, 52) | fault runs 10


---

## Task 1. The test stream and the unit cube  
*0.10 points: 0.05 for the code (tests), 0.05 for the answers*

Each variable $j$ is mapped to the unit interval with the range of the **normal training data
only**, and the normal data are placed in the middle half of the cube:

$$\text{lo}_j = \min_i x_{ij}, \quad \text{hi}_j = \max_i x_{ij}, \qquad
u_j = 0.25 + 0.5\,\frac{x_j - \text{lo}_j}{\text{hi}_j - \text{lo}_j}, \quad \text{clipped to } [0, 1].$$

Normal training data fill $[0.25, 0.75]$, and the outer band is room for non-self. A reading
beyond the band is clipped to the face of the cube.

Why should 500 samples miss part of the normal range? For a continuous variable, a new sample
falls outside $[\min, \max]$ of $n$ earlier samples from the same distribution with probability
$2/(n+1)$, about 0.4 % for $n = 500$. With 52 independent variables a new row stays inside on
every one of them with probability about $(1 - 2/501)^{52} \approx 0.81$. Real variables are
correlated and drift between runs, so the real share can differ a lot.

**Write**

- `fit_box(X)`: `(lo, hi)`, the column-wise minimum and maximum of a 2-D array, two 1-D arrays;
- `to_unit(X, lo, hi)`: the rule above for every row of `X`, a new float array of the shape of
  `X`, clipped to $[0, 1]$;
- `outside_share`: a `float`, the share of the rows of `X_normal` that have at least one variable
  outside the box $[\text{lo}, \text{hi}]$ of `X_train`.

In [ ]:
def fit_box(X):
    ...


def to_unit(X, lo, hi):
    ...


outside_share = ...

In [ ]:
# Tests for Task 1 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_box_of_three_rows():
    lo, hi = fit_box(np.array([[0., 10.], [2., 30.], [4., 20.]]))
    assert np.allclose(lo, [0, 10]) and np.allclose(hi, [4, 30]), f"lo {lo}, hi {hi}"


test_box_of_three_rows()


def test_unit_cube_with_clipping():
    X = np.array([[2., 20.], [8., 0.], [1., 25.]])
    got = to_unit(X, np.array([0., 10.]), np.array([4., 30.]))
    assert isinstance(got, np.ndarray) and got.shape == (3, 2)
    msg = ("the middle of the box goes to 0.5, the edges to 0.25 and 0.75, "
           f"beyond is clipped; got {got.tolist()}")
    assert np.allclose(got, [[0.5, 0.5], [1.0, 0.0], [0.375, 0.625]]), msg


test_unit_cube_with_clipping()


def test_outside_share_is_a_share():
    ok = isinstance(outside_share, float) and 0 <= outside_share <= 1
    assert ok, "outside_share: a float between 0 and 1"


test_outside_share_is_a_share()


print("all tests passed")

**Questions** (answer them in `practice_8_answers.md`, Task 1, with the numbers your code printed):

1. Which file is stored on its side, and how does the loader notice? How many hours does each file cover?
2. Standardise every run with the mean and std of `X_train`. Plot the mean of $|z|$ over the 52 variables against the sample number for the runs of faults 1 and 9, with a vertical line at the onset. When does fault 1 become visible? Fault 9?
3. Report `outside_share` and the worst variable. What does this say about 500 samples as a picture of "self", and why do we leave a margin around the box?
4. The first 160 samples of each test run are normal too. Why do we not add them to the training data?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 2. Negative selection in two variables  
*0.20 points: 0.10 for the code (tests), 0.10 for the answers*

The **self set** $S$ is the training data in unit coordinates, points $\mathbf{s}_i$.

| step | rule |
|---|---|
| self radius | $r_s$ = the $q$-quantile of $\min_i \lVert \mathbf{v} - \mathbf{s}_i \rVert$ over the held-out normal points $\mathbf{v}$; with $q = 0.99$, 99 % of normal points lie within $r_s$ of a self point |
| censoring | a candidate $\mathbf{c}$ is **deleted** if $\min_i \lVert \mathbf{c} - \mathbf{s}_i \rVert \le r_s$; the others are kept as detectors $\mathbf{d}_j$ |
| matching | a point $\mathbf{x}$ is **abnormal** if $\min_j \lVert \mathbf{x} - \mathbf{d}_j \rVert \le r_d$ |
| score | $a(\mathbf{x}) = r_d - \min_j \lVert \mathbf{x} - \mathbf{d}_j \rVert$; $a(\mathbf{x}) \ge 0$ exactly when $\mathbf{x}$ is matched |

The detectors never see a fault. The self radius is a data-driven answer to "how close is
normal": instead of guessing $r_s$, we take the distance within which almost all *new* normal
points fall. A larger score means "closer to a detector". Moving the threshold on $a$ is the same
as moving $r_d$, and this is what the ROC in Task 3 does.

`dist_matrix(A, B)` from the loader cell gives all the distances at once.

**Write**

- `self_radius(S, V, q=0.99)`: $r_s$ as a `float` (`np.quantile` with its default interpolation);
- `censor(C, S, r_s)`: the rows of `C` farther than `r_s` from every row of `S`, in their order;
- `ns_score(X, D, r_d)`: the score $a$ of every row of `X`, a 1-D array.

The common cell under the tests builds detectors in the plane of the A feed `XMEAS1` and the
reactor cooling water valve `XMV10`. After it has run, write in the cell for the questions:

- `pair_rates`: a `pd.DataFrame` indexed by `FAULTS` with the columns `"detected after onset"` and
  `"false alarms before"`: the share of samples of each fault run with
  `ns_score(unit2(RUNS[f]), D2, r_s2) >= 0` after and before `ONSET`;
- `normal_matched`: a `float`, the same share on all of `X_normal`;
- `uncovered`: a dict `{5000: ..., 300: ...}`. Draw 20 000 probes with
  `np.random.default_rng(5).random((20000, 2))`; those farther than `r_s2` from every self point
  are non-self. The values are the shares of non-self probes not matched (with $r_d$ = `r_s2`) by
  the detectors `D2` and by the detectors from the first 300 candidates, `censor(C2[:300], S2, r_s2)`.

In [ ]:
def self_radius(S, V, q=0.99):
    ...


def censor(C, S, r_s):
    ...


def ns_score(X, D, r_d):
    ...

In [ ]:
# Tests for Task 2 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_self_radius_by_hand():
    S = np.array([[0.4, 0.4], [0.6, 0.4], [0.5, 0.6]])
    V = np.array([[0.45, 0.4], [0.5, 0.5], [0.7, 0.7]])
    got = self_radius(S, V, q=0.5)
    ok = isinstance(got, float) and abs(got - 0.1) < 1e-9
    assert ok, f"nearest-self distances 0.05, 0.1, 0.224: the median is 0.1, got {got}"


test_self_radius_by_hand()


def test_censoring_keeps_the_far_candidates():
    S = np.array([[0.4, 0.4], [0.6, 0.4], [0.5, 0.6]])
    C = np.array([[0.1, 0.1], [0.5, 0.45], [0.9, 0.5], [0.5, 0.9]])
    got = np.asarray(censor(C, S, 0.15)).tolist()
    msg = f"the second candidate lies within 0.15 of self; got {got}"
    assert got == [[0.1, 0.1], [0.9, 0.5], [0.5, 0.9]], msg


test_censoring_keeps_the_far_candidates()


def test_score_is_rd_minus_the_nearest_distance():
    D = np.array([[0.1, 0.1], [0.9, 0.5], [0.5, 0.9]])
    got = ns_score(np.array([[0.9, 0.6], [0.5, 0.5], [0.1, 0.35]]), D, 0.2)
    assert np.allclose(got, [0.1, -0.2, -0.05]), f"got {np.round(got, 4)}"


test_score_is_rd_minus_the_nearest_distance()


print("all tests passed")

*Common ground: run this cell as it is, after your tests pass.*

In [ ]:
PAIR = ["XMEAS1", "XMV10"]     # A feed (stream 1) and the reactor cooling water valve
cols = [VARS.index(v) for v in PAIR]

lo2, hi2 = fit_box(X_train[:, cols])
unit2 = lambda X: to_unit(X[:, cols], lo2, hi2)
S2 = unit2(X_train)
r_s2 = self_radius(S2, unit2(X_normal))
C2 = draw_candidates(2)
D2 = censor(C2, S2, r_s2)
print(f"r_s = {r_s2:.4f}; candidates {len(C2)}, deleted {len(C2) - len(D2)}, "
      f"detectors kept {len(D2)}")

**Questions** (answer them in `practice_8_answers.md`, Task 2, with the numbers your code printed):

1. How many candidates were deleted? Plot the unit square: the self points, the detectors, and the samples of the runs of faults 1 and 4 after the onset.
2. Report `pair_rates` and `normal_matched`. Which faults does this pair of variables see? Why is it blind to faults 2 and 7?
3. Why are there false alarms on normal data, although 99 % of normal points lie within $r_s$ of self?
4. Report `uncovered` and show the uncovered probes of the 300-candidate detectors on a second panel. Does negative selection guarantee that every abnormal state is covered?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 3. The ROC curve and its area  
*0.15 points: 0.08 for the code (tests), 0.07 for the answers*

For a threshold $t$ every sample with score $\ge t$ is called abnormal. With $P$ abnormal and $N$
normal samples,

$$\text{TPR}(t) = \frac{\#\{\text{abnormal},\ a \ge t\}}{P}, \qquad \text{FPR}(t) = \frac{\#\{\text{normal},\ a \ge t\}}{N}.$$

The ROC curve joins the points $(\text{FPR}, \text{TPR})$ for every distinct score taken as $t$, from
the largest to the smallest, starting at $(0, 0)$. Its area by the trapezoid rule:

$$\text{AUC} = \sum_k (\text{FPR}_{k+1} - \text{FPR}_k)\, \frac{\text{TPR}_{k+1} + \text{TPR}_k}{2}.$$

**AUC is a ranking probability.** Pick one abnormal sample and one normal sample at random. The
AUC is the probability that the abnormal one scores higher, ties counted as one half:

$$\text{AUC} = P(a_{\text{abnormal}} > a_{\text{normal}}) + \tfrac12 P(a_{\text{abnormal}} = a_{\text{normal}}).$$

Why the area equals this: lowering $t$ past one normal sample moves the curve right by $1/N$, and
the height of the curve there is the share of abnormal samples that score above that normal
one. Summing height times width over all normal samples counts the abnormal-above-normal pairs,
divided by $PN$. So AUC depends only on the *order* of the scores, never on their scale: any
increasing transformation of a score leaves it unchanged.

**Write**

- `roc_curve(scores, labels)`: `(fpr, tpr)`, two 1-D arrays of the same length: a first point
  $(0, 0)$, then one point per distinct score, largest first; `labels` are 1 for abnormal and 0
  for normal;
- `auc(fpr, tpr)`: the trapezoid sum as a `float`, written out (no `np.trapz`);
- `AUC2`: a dict fault $\to$ the AUC of the 2-D detectors of Task 2 on the full run,
  `ns_score(unit2(RUNS[f]), D2, r_s2)` against `LABEL`.

In [ ]:
def roc_curve(scores, labels):
    ...


def auc(fpr, tpr):
    ...


AUC2 = ...

In [ ]:
# Tests for Task 3 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_six_samples_by_hand():
    scores = np.array([0.9, 0.7, 0.7, 0.4, 0.3, 0.1])
    labels = np.array([1, 0, 1, 1, 0, 0])
    fpr, tpr = roc_curve(scores, labels)
    assert np.allclose(fpr, [0, 0, 1/3, 1/3, 2/3, 1]), f"fpr {np.round(fpr, 3)}"
    msg = f"tpr {np.round(tpr, 3)}: the tie at 0.7 is one threshold"
    assert np.allclose(tpr, [0, 1/3, 2/3, 1, 1, 1]), msg


test_six_samples_by_hand()


def test_trapezoid_area():
    got = auc(np.array([0, 0, 1/3, 1/3, 2/3, 1]), np.array([0, 1/3, 2/3, 1, 1, 1]))
    assert isinstance(got, float) and abs(got - 5 / 6) < 1e-9, f"expected 5/6, got {got}"
    diagonal = auc(np.array([0.0, 1.0]), np.array([0.0, 1.0]))
    assert abs(diagonal - 0.5) < 1e-12, "the diagonal has area 0.5"


test_trapezoid_area()


def test_auc2_has_every_fault():
    assert isinstance(AUC2, dict) and set(AUC2) == set(FAULTS)
    assert all(0 <= v <= 1 for v in AUC2.values())


test_auc2_has_every_fault()


print("all tests passed")

**Questions** (answer them in `practice_8_answers.md`, Task 3, with the numbers your code printed):

1. Plot the ten ROC curves on one figure and print `AUC2`. Check two of your AUCs against `sklearn.metrics.roc_auc_score`.
2. Mark on the curve of fault 1 the operating point of Task 2, $r_d = r_s$, and report its coordinates.
3. Which faults have AUC near 0.5? What does an AUC of 0.5 mean for an operator, and what would an AUC well below 0.5 mean?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 4. 52 variables, PCA and the curse of dimensionality  
*0.20 points: 0.10 for the code (tests), 0.10 for the answers*

PCA on the **normal training data only**. Standardise with the mean and std (population, numpy
default) of `X_train`, stack the rows into $\mathbf{Z}$ ($n \times p$), and take its SVD,
$\mathbf{Z} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^\top$. The columns of $\mathbf{V}$ are the
principal directions, sorted by $\sigma_m^2$; $\mathbf{V}_k$ holds the first $k$ of them.

| quantity | formula |
|---|---|
| explained share of component $m$ | $\sigma_m^2 / \sum_l \sigma_l^2$ |
| scores of a point $\mathbf{z}$ | $\mathbf{t} = \mathbf{V}_k^\top \mathbf{z}$, the first $k$ directions |
| residual | $\mathbf{e} = \mathbf{z} - \mathbf{V}_k \mathbf{t} = (\mathbf{I} - \mathbf{V}_k\mathbf{V}_k^\top)\,\mathbf{z}$ |
| SPE (squared prediction error) | $Q = \lVert \mathbf{e} \rVert^2$; we use $\log_{10} Q$ as one more feature |

**The geometry.** Normal operation fills, roughly, a $k$-dimensional plane through the mean: the
span of $\mathbf{V}_k$. $\mathbf{P} = \mathbf{V}_k\mathbf{V}_k^\top$ projects a point onto that
plane. The scores $\mathbf{t}$ say *where on the plane* the point sits, and $Q$ is its squared
**distance to the plane**. By Pythagoras, $\lVert\mathbf{z}\rVert^2 = \lVert\mathbf{t}\rVert^2 + Q$:
every standardised point splits into a part along normal variation and a part off it. The scores
show movement *along* normal variation, and $Q$ shows movement *off* it.

**Write**

- `pca_fit(X)`: a dict with `"mu"` and `"sd"` (the column means and stds of `X`), `"V"` (the
  $p \times p$ matrix whose **rows** are the directions, the strongest first: `Vt` of
  `np.linalg.svd(Z, full_matrices=False)`) and `"explained"` (the $p$ shares);
- `pca_features(X, model, k, spe=True)`: for raw rows `X`, the $n \times k$ scores, plus
  $\log_{10} Q$ as an extra last column when `spe` is true.

The common cell under the tests fits `PCA = pca_fit(X_train)` and defines `ns_pipeline`, `auc_row`
and the six feature sets `FEATURE_SETS`. After it has run, write in the cell for the questions:

- `k90`: an `int`, the number of components of `PCA` needed for 90 % of the variance;
- `T4`: a `pd.DataFrame` indexed by the names of `FEATURE_SETS`, one row per set: the `info` of
  `ns_pipeline(F)` (columns `"dim"`, `"r_s"`, `"kept"`, `"median nearest self, candidates"`) and the
  AUC of constant-radius negative selection, $r_d = r_s$, for every fault (columns `FAULTS`):
  `auc_row(lambda X: ns_score(unit(X), D, r_s))`.

In [ ]:
def pca_fit(X):
    ...


def pca_features(X, model, k, spe=True):
    ...

In [ ]:
# Tests for Task 4 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_shares_of_a_small_set():
    model = pca_fit(np.array([[2., 2.], [-2., -2.], [1., -1.], [-1., 1.]]))
    ok = isinstance(model, dict) and {"mu", "sd", "V", "explained"} <= set(model)
    assert ok, "a dict with mu, sd, V, explained"
    assert np.allclose(model["explained"], [0.8, 0.2]), f"explained {model['explained']}"


test_shares_of_a_small_set()


def test_score_and_spe_of_one_point():
    model = pca_fit(np.array([[2., 2.], [-2., -2.], [1., -1.], [-1., 1.]]))
    F = pca_features(np.array([[2., 1.]]), model, k=1)
    assert F.shape == (1, 2), "k scores plus the SPE column"
    ok = abs(abs(F[0, 0]) - 1.3416) < 1e-4 and abs(F[0, 1] + 0.699) < 1e-3
    assert ok, f"score +-1.3416 and log10 SPE -0.699, got {F[0].round(4)}"
    scores_only = pca_features(np.array([[2., 1.]]), model, k=1, spe=False)
    assert scores_only.shape == (1, 1), "no SPE column when spe is False"


test_score_and_spe_of_one_point()


print("all tests passed")

*Common ground: run this cell as it is, after your tests pass.*

In [ ]:
def ns_pipeline(F, n_cand=5000):
    # F maps raw rows (n x 52) to features. Box, self radius and detectors come from
    # normal data only. Returns (unit: raw rows -> unit cube, S, r_s, D, info); the
    # candidates are draw_candidates(dim).
    lo, hi = fit_box(F(X_train))
    unit = lambda X: to_unit(F(X), lo, hi)
    S = unit(X_train)
    r_s = self_radius(S, unit(X_normal))
    C = draw_candidates(S.shape[1], n=n_cand)
    D = censor(C, S, r_s)
    info = {"dim": S.shape[1], "r_s": r_s, "kept": len(D),
            "median nearest self, candidates": float(np.median(dist_matrix(C, S).min(1)))}
    return unit, S, r_s, D, info


def auc_row(score_fn):
    # AUC of a score function (raw rows -> score) on every fault run.
    return {f: auc(*roc_curve(score_fn(RUNS[f]), LABEL)) for f in FAULTS}


FEATURE_SETS = {"52 variables": lambda X: X}
for _k in (2, 3, 5, 10):
    FEATURE_SETS[f"PCA k={_k}"] = lambda X, k=_k: pca_features(X, PCA, k, spe=False)
FEATURE_SETS["PCA k=3 + SPE"] = lambda X: pca_features(X, PCA, 3, spe=True)

PCA = pca_fit(X_train)
print("PCA fitted on X_train:", PCA["V"].shape, "| feature sets:", list(FEATURE_SETS))

**Questions** (answer them in `practice_8_answers.md`, Task 4, with the numbers your code printed):

1. How much variance do the first 2, 3, 5 and 10 components explain? Report `k90`.
2. Print `T4`. What happens with 52 variables? Use the numbers of the table: $r_s$, how many candidates were deleted, how far the candidates are from self.
3. Which faults gain from the SPE column? What do these faults do to the plant, and why are they hard to see in the first three components?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 5. V-detectors: a radius for every detector  
*0.15 points: 0.07 for the code (tests), 0.08 for the answers*

A V-detector (Ji and Dasgupta, 2004) keeps the candidate's own distance to self. For a candidate
$\mathbf{c}$ with $\delta(\mathbf{c}) = \min_i \lVert \mathbf{c} - \mathbf{s}_i \rVert$:

| step | rule |
|---|---|
| censoring | as before: deleted if $\delta(\mathbf{c}) \le r_s$ |
| radius | $\rho_j = \delta(\mathbf{d}_j) - r_s > 0$: the detector reaches up to $r_s$ from the nearest self point |
| matching | $\mathbf{x}$ is abnormal if $\lVert \mathbf{x} - \mathbf{d}_j \rVert \le \rho_j$ for some $j$ |
| score | $a_V(\mathbf{x}) = \max_j \big(\rho_j - \lVert \mathbf{x} - \mathbf{d}_j \rVert\big)$; $a_V(\mathbf{x}) \ge 0$ exactly when $\mathbf{x}$ is matched |

A detector far from self gets a large radius and covers a lot of empty space. Near self the radii
shrink, so the border of self is still drawn finely. Each ball is the largest one around its
centre that keeps a margin of $r_s$ from every self point, so the V-detectors fill non-self
without ever touching the self zone.

**Write**

- `v_detectors(C, S, r_s)`: `(D, rho)`, the kept candidates in their order and their radii;
- `v_score(X, D, rho)`: the score $a_V$ of every row of `X`, a 1-D array.

The common cell under the tests fixes the features for everybody (PCA $k = 3$ plus SPE): `unit4`,
`S4`, `r_s4`, the constant-radius detectors `D4`, the candidates `C4` and 20 000 probes `PROBES`.
After it has run, write in the cell for the questions:

- `coverage_table`: a `pd.DataFrame` indexed by `"4-D, 5000 candidates"`, `"4-D, 500 candidates"`,
  `"10-D, 5000 candidates"`, `"10-D, 500 candidates"` with the columns `"non-self probes"`,
  `"constant radius"`, `"V-detectors"`. Non-self probes lie farther than $r_s$ from every self
  point; a probe is covered by constant radius if it lies within $r_s$ of a detector of
  `censor(C, S, r_s)`, and by V-detectors if its `v_score` is $\ge 0$. 4-D uses `S4`, `r_s4`, `C4`
  (or `C4[:500]`) and `PROBES`; 10-D uses `ns_pipeline` on PCA $k = 10$ without SPE,
  `draw_candidates(10)` and probes `np.random.default_rng(29).random((20000, 10))`;
- `T5`: a `pd.DataFrame` of AUCs, columns `FAULTS`, rows `"constant, k=3+SPE"`, `"V, k=3+SPE"`,
  `"constant, k=10"`, `"V, k=10"`, with V-detectors from the same 5000 candidates.

In [ ]:
def v_detectors(C, S, r_s):
    ...


def v_score(X, D, rho):
    ...

In [ ]:
# Tests for Task 5 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_detectors_and_radii_by_hand():
    S = np.array([[0.4, 0.4], [0.6, 0.4], [0.5, 0.6]])
    C = np.array([[0.1, 0.1], [0.5, 0.45], [0.9, 0.5], [0.5, 0.9]])
    D, rho = v_detectors(C, S, 0.15)
    kept = np.asarray(D).tolist()
    assert kept == [[0.1, 0.1], [0.9, 0.5], [0.5, 0.9]], "the same censoring as before"
    msg = f"radius = distance to self minus r_s; got {np.round(rho, 4)}"
    assert np.allclose(rho, [0.27426, 0.16623, 0.15], atol=1e-4), msg


test_detectors_and_radii_by_hand()


def test_score_of_three_points():
    D = np.array([[0.1, 0.1], [0.9, 0.5], [0.5, 0.9]])
    rho = np.array([0.27426, 0.16623, 0.15])
    got = v_score(np.array([[0.9, 0.6], [0.5, 0.5], [0.1, 0.35]]), D, rho)
    ok = np.allclose(got, [0.06623, -0.23377, 0.02426], atol=1e-4)
    assert ok, f"got {np.round(got, 4)}"


test_score_of_three_points()


print("all tests passed")

*Common ground: run this cell as it is, after your tests pass.*

In [ ]:
FEAT = lambda X: pca_features(X, PCA, 3, spe=True)    # the same features for everybody
unit4, S4, r_s4, D4, info4 = ns_pipeline(FEAT)
C4 = draw_candidates(S4.shape[1])                     # what ns_pipeline censored
PROBES = np.random.default_rng(29).random((20000, S4.shape[1]))
print(info4)

**Questions** (answer them in `practice_8_answers.md`, Task 5, with the numbers your code printed):

1. How many V-detectors are kept on the common features, and what are the smallest, median and largest radius? Compare with $r_s$.
2. Report `coverage_table`. Where does the constant radius cover more, where the V-detectors?
3. Report `T5`. Why does a variable radius help more as the dimension grows? Which statement of the lecture about coverage do your numbers support?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

*Common ground for Tasks 6 and 7: V-detectors on the common features and their score. Run it as it is.*

In [ ]:
Dv4, rho4 = v_detectors(C4, S4, r_s4)
score_v = lambda X: v_score(unit4(X), Dv4, rho4)          # raw rows -> V-score
print(f"V-detectors on the common features: {len(Dv4)}")

---

## Task 6. From a score to an alarm: threshold and detection delay  
*0.20 points: 0.10 for the code (tests), 0.10 for the answers*

**Threshold.** Sort the scores of the held-out normal run from the largest down,
$a_{(1)} \ge a_{(2)} \ge \dots \ge a_{(n)}$. For a false-alarm rate $\alpha$ take

$$t = a_{(\lfloor \alpha n \rfloor + 1)}, \qquad \text{alarm if } a(\mathbf{x}) > t.$$

At most $\lfloor \alpha n \rfloor$ normal samples lie strictly above $t$. The threshold is an
empirical quantile, so it is set by the few most extreme normal samples: with $n = 960$ and
$\alpha = 1\,\%$, by the top 9.

**Detection delay.** One sample over the threshold is not an alarm yet. The alarm is **confirmed**
at the first sample that completes `confirm` consecutive samples over the threshold, counting from
the onset. The delay is the number of samples from the onset to that sample:

$$\text{delay} = i_{\text{alarm}} - \text{onset}, \qquad \text{the smallest possible delay is } \text{confirm} - 1.$$

If normal samples crossed the threshold independently with probability $\alpha$, a run of `confirm`
crossings would start at a given sample with probability $\alpha^{\text{confirm}}$. Real scores
are correlated in time, so normal excursions come in runs and confirmation helps less than this
suggests.

**Write**

- `threshold_at_far(s_normal, far)`: $t$ as a `float`;
- `detection_delay(flags, onset, confirm=3)`: the delay in samples as an `int`, or `None` if no
  alarm is confirmed; only `flags[onset:]` count, so a run that started before the onset counts
  from the onset;
- `thr`: the threshold at FAR 1 % on `score_v(X_normal)`;
- `T6`: a `pd.DataFrame` indexed by `FAULTS` with the columns `"AUC"` (of `score_v` on the run
  against `LABEL`), `"false alarms before"` and `"detected after"` (the shares of samples with
  score $>$ `thr` before and after `ONSET`) and `"delay, min"` (with `confirm = 3`, in minutes,
  NaN if missed).

In [ ]:
def threshold_at_far(s_normal, far):
    ...


def detection_delay(flags, onset, confirm=3):
    ...


thr = ...
T6 = ...

In [ ]:
# Tests for Task 6 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_threshold_of_ten_scores():
    s = np.array([5., 1., 4., 2., 3., 9., 7., 8., 6., 10.])
    got = threshold_at_far(s, 0.2)
    ok = isinstance(got, float) and got == 8.0
    assert ok, f"two of ten may lie above: t = 8, got {got}"
    top = threshold_at_far(s, 0.05)
    assert top == 10.0, "floor(0.5) = 0 samples above: t is the largest"


test_threshold_of_ten_scores()


def test_delay_with_confirmation():
    fl = np.array([0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0], bool)
    got = detection_delay(fl, onset=4, confirm=3)
    assert got == 5, "three in a row are completed at index 9: delay 5"
    assert detection_delay(fl, onset=4, confirm=2) == 1
    assert detection_delay(fl, onset=4, confirm=5) is None, "no run of five: None"
    assert detection_delay(fl, onset=0, confirm=1) == 1


test_delay_with_confirmation()


def test_results_have_the_right_type():
    assert isinstance(thr, float)
    assert isinstance(T6, pd.DataFrame) and list(T6.index) == FAULTS
    needed = {"AUC", "false alarms before", "detected after", "delay, min"}
    assert needed <= set(T6.columns)


test_results_have_the_right_type()


print("all tests passed")

**Questions** (answer them in `practice_8_answers.md`, Task 6, with the numbers your code printed):

1. Report `thr` and `T6`. Which faults are missed?
2. Plot the score against the sample number for faults 1, 4 and 9 and for `X_normal`, with the threshold and the onset. Where do the top 1 % of `X_normal` come from?
3. Compare the AUC of fault 4 with its delay. Why can a well-ranked fault alarm so late? Fault 15 gets an alarm after more than a day: is it detected?
4. Try FAR = 1 % and 5 %, each with `confirm` = 1, 3 and 6. Print for each pair: the number of faults 1, 2, 4, 6, 7 caught and their mean delay, the delay of fault 4 and of fault 9, and the number of runs with a confirmed alarm **before** the onset. Is the early alarm of fault 9 at 5 % a detection? Which setting would you put into a control room?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

## Task 7. Isolation Forest, One-Class SVM and the engineering comparison  
*0.25 points: 0.10 for the code (tests), 0.15 for the answers*

One function turns the scores of any detector into the row of an engineering table. For every
run: AUC on the full run; the threshold at the given FAR on the normal run; the false-alarm rate
before the onset; the detection rate after it; the delay in samples with the confirmation rule; and
**missed** = no confirmed alarm after the onset.

AUC and delay answer different questions. AUC compares the faulty part of a run with its own
normal part, over all thresholds. The delay depends on one threshold, set on another run, and on
how far above it the faulty scores go. A detector can rank perfectly and still alarm late.

**Write**

- `summarise(scores_by_run, s_normal, onset, far=0.01, confirm=3)`: `scores_by_run` is a dict
  name $\to$ score array of one run, whose samples from `onset` on are abnormal. Returns a
  `pd.DataFrame` with one row per run, in the order of the dict, and the columns `"AUC"`,
  `"false alarms before"`, `"detected after"` (floats), `"delay"` (samples, NaN if missed) and
  `"missed"` (bool), with the threshold `threshold_at_far(s_normal, far)` and alarm if score $>$
  threshold;
- `auc_tab` and `delay_tab`: `pd.DataFrame`s indexed by `FAULTS`, one column per detector, the
  `"AUC"` and the `"delay"` **in minutes** of `summarise({f: score(RUNS[f]) for f in FAULTS},
  score(X_normal), ONSET)` (FAR 1 %, `confirm = 3`). The six detectors, by column name:
  `"NS constant"` (`ns_score(unit4(X), D4, r_s4)`), `"NS V-det"` (`score_v`), and Isolation Forest
  and One-Class SVM on the common features and on the 52 raw variables: `"IF, 4 feat"`,
  `"OCSVM, 4 feat"`, `"IF, 52 var"`, `"OCSVM, 52 var"`. The baselines learn from `X_train` only:
  standardise their inputs with the mean and std of the training features, fit
  `IsolationForest(n_estimators=200, random_state=0)` and `OneClassSVM(nu=0.05, gamma="scale")`,
  and score with `-model.score_samples(Z)` and `-model.decision_function(Z)`, so that a larger
  score means "more abnormal".

In [ ]:
def summarise(scores_by_run, s_normal, onset, far=0.01, confirm=3):
    ...


auc_tab = ...
delay_tab = ...

In [ ]:
# Tests for Task 7 (public). They check what your code does, not how it is written.
# Run this cell after your code; an AssertionError says which check failed.


def test_two_tiny_runs():
    # normal scores 0.0 ... 0.9; at FAR 10 % the threshold is 0.8
    s_norm = np.arange(10, dtype=float) / 10
    runs = {"a": np.array([0.1, 0.2, 0.6, 0.95, 0.99, 0.97, 0.5]),
            "b": np.array([0.95, 0.0, 0.1, 0.2, 0.3, 0.1, 0.2])}
    got = summarise(runs, s_norm, onset=3, far=0.1, confirm=2)
    assert isinstance(got, pd.DataFrame), "summarise returns a DataFrame"
    assert list(got.index) == ["a", "b"], "one row per run, in order"
    assert np.allclose(got["AUC"], [11 / 12, 0.625])
    assert np.allclose(got["false alarms before"], [0, 1 / 3])
    ok = np.allclose(got["detected after"], [0.75, 0])
    assert ok, "a: 3 of 4 samples after the onset are over 0.8"
    assert got.loc["a", "delay"] == 1 and np.isnan(got.loc["b", "delay"])
    assert got["missed"].tolist() == [False, True]


test_two_tiny_runs()


def test_tables_have_six_detectors():
    cols = {"NS constant", "NS V-det", "IF, 4 feat", "OCSVM, 4 feat",
            "IF, 52 var", "OCSVM, 52 var"}
    for name, tab in [("auc_tab", auc_tab), ("delay_tab", delay_tab)]:
        assert isinstance(tab, pd.DataFrame), f"{name}: a DataFrame"
        shape_ok = set(tab.columns) == cols and set(tab.index) == set(FAULTS)
        assert shape_ok, f"{name}: faults x the six detectors"


test_tables_have_six_detectors()


print("all tests passed")

**Questions** (answer them in `practice_8_answers.md`, Task 7, with the numbers your code printed):

1. Try `nu` = 0.01, 0.05, 0.2 for the SVM and `contamination` = "auto", 0.01, 0.1 for Isolation Forest on the common features. Which of them changes the AUC, which does not, and why?
2. Print `auc_tab` and `delay_tab`, and per detector: the number of missed faults, the mean AUC over faults 1, 2, 4, 5, 6, 7, 11, and the false-alarm rate before the onset over all runs. Plot the ROC curves of the six detectors for fault 4.
3. Constant radius and V-detectors have almost the same AUC. Why are their delays so different?
4. Which faults are caught by every method, which by none? For faults 3, 9 and 15, is a better detector the answer, or is something missing in the data?
5. **Conclusion**, 6–10 sentences for a plant engineer: which detector would you install, and on which features? What does negative selection give that the baselines do not, and what does it cost?

In [ ]:
# your code for the questions: tables, figures, anything you quote in the answers

---

### Sources

- Downs J.J., Vogel E.F. A plant-wide industrial process control problem. Computers & Chemical
  Engineering 17 (3), 1993. (the process)
- Chiang L.H., Russell E.L., Braatz R.D. Fault Detection and Diagnosis in Industrial Systems.
  Springer, 2001. (the data files; faults 3, 9 and 15 as hard to detect)
- Forrest S., Perelson A.S., Allen L., Cherukuri R. Self-nonself discrimination in a computer.
  Proc. IEEE Symposium on Research in Security and Privacy, 1994. (negative selection)
- Ji Z., Dasgupta D. Real-valued negative selection algorithm with variable-sized detectors.
  Proc. GECCO, 2004. (V-detectors)
- Liu F.T., Ting K.M., Zhou Z.-H. Isolation forest. Proc. ICDM, 2008.
- Schölkopf B. et al. Estimating the support of a high-dimensional distribution. Neural
  Computation 13 (7), 2001. (One-Class SVM)